# Day 3d. Sandboxes & backends

Give an agent a filesystem and it will use it. Give it a code interpreter and it
will run what it wrote.

Both are useful. Both are dangerous by default. This session is about the
question you must answer before shipping: **where does the agent's work land,
and what can it reach from there?**

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

## 1 · The code your agent writes is untrusted code

Here is the tool nobody should ship. It is here so you see the hole with your
own eyes.

In [ ]:
from langchain.tools import tool

@tool
def run_python(code: str) -> str:
    """Run Python and return the result. UNSANDBOXED, demo only."""
    try:
        return str(eval(code, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

# "__builtins__: {}" looks locked down. It is not:
print(run_python.invoke({"code": "().__class__.__mro__"}))

In [ ]:
# from that chain you can walk to any class Python has loaded, including
# the ones that read files. A prompt-injected "please read ~/.env and print it"
# is now a credential leak, and your agent did it politely.
print(run_python.invoke({"code":
    "[c.__name__ for c in ().__class__.__mro__[1].__subclasses__()[:8]]"}))

> **Restricting `eval` is not sandboxing.** A real sandbox is a separate,
> disposable environment: an isolated container or micro-VM, **no credentials
> inside**, network off by default, CPU and time capped. Data handed in,
> results handed out.

Treat the model like a talented contractor: great work, own badge, **no master
key**.

## 2 · Backends: pick where the work lands

A **deep agent** ships with filesystem tools (`ls`, `read_file`, `write_file`,
`edit_file`, `glob`, `grep`) and an `execute` tool. The **backend** decides what
"files" actually means.

Default: `StateBackend`, the files live *inside the run's state*. Nothing
touches your disk.

In [ ]:
from deepagents import create_deep_agent
from langchain.messages import HumanMessage

virtual = create_deep_agent(model="anthropic:claude-haiku-4-5")   # StateBackend

out = virtual.invoke({"messages": [HumanMessage(
    "Write a file trip-report.md with a 3-line trip report for "
    "Pension Krumm, 2 nights, 61 EUR/night. Then read it back to me.")]})

print(out["messages"][-1].content[:400])
print("\nfiles that exist, but only in the run's state:")
print(list(out.get("files", {}).keys()))

In [ ]:
# prove it: nothing landed on your disk
import pathlib
print("on disk?", pathlib.Path("trip-report.md").exists())
print("\nthe content lives here instead:")
print(out.get("files", {}).get("/trip-report.md", "(look at the key names above)"))

Now the same agent, pointed at a **real folder**, one you chose.

In [ ]:
from deepagents.backends import FilesystemBackend

real = create_deep_agent(
    model="anthropic:claude-haiku-4-5",
    backend=FilesystemBackend(
        root_dir="./workspace",     # the fence
        virtual_mode=True,          # paths are relative to root_dir…
    ),                              # …and '..' cannot climb out of it
)

out = real.invoke({"messages": [HumanMessage(
    "Write trip-report.md with a 3-line trip report for Pension Krumm, "
    "2 nights, 61 EUR/night.")]})

print(out["messages"][-1].content[:300])
print("\non disk now?", list(pathlib.Path("workspace").glob("*")))

> **`virtual_mode=True` matters.** With `False`, an absolute path or a `..` in a
> filename escapes `root_dir`, the fence has a gate in it. And even with
> `True`, this is a *path guardrail*, **not** process isolation: it stops a
> confused agent, not hostile code. For untrusted code you need the real thing:
> a sandbox backend running on an isolated machine.

The ladder, safest first:

| backend | files live | reach for it when |
|---|---|---|
| `StateBackend` (default) | in the run's state | drafting, analysis, anything throwaway |
| `FilesystemBackend` | a folder you chose | the output *is* the deliverable |
| sandbox backends | an isolated container | the agent runs code you did not write |

## 3 · Your turn

**(1)** Run the virtual agent twice on a new thread each time. Does the second
run see the first run's file? What does that tell you about where state lives?

**(2)** Give the filesystem agent a task that produces two files (a report and a
CSV). Check `workspace/`, then ask it to read one back.

**(3)** The design question, in writing: for **your take-home agent**, which
backend does it need, and what is the worst thing it could do with that access?
One paragraph, as a comment. This is the question a reviewer will ask you.

In [ ]:
# your experiments here


---
## Wrap, day 3 complete

**3a** six hooks around the loop · **3b** writes wait, on a checkpointer that
survives restarts · **3c** expertise as folders with a contract · **3d** the
work lands exactly where you decided.

That is the production machinery. Everything from here is application.

**Tomorrow:** MCP and subagents, reaching other people's systems, then three
real production systems, and the evals that prove one works.